# Crate architecture and the pipeline

`onceover` is a Cargo workspace of six crates with one direction of
dependencies, plus a minimal Python wrapper bound by
[`pyo3`](https://github.com/PyO3/pyo3) and built with maturin. Every
profiling algorithm, spec inference, multi-table foreign-key analysis,
report rendering (JSON, compact/budgeted JSON, Markdown, digest, brief),
lookup, filtering, and diffing lives in Rust so Python (and future language
bindings such as R) stays thin and behaviorally identical.

```
onceover-core      typed column extraction from polars, exact value
                   counting, column cards, duplicates, date utilities
onceover-deps      keys and exact relationships: FD/AFD discovery,
                   unique column combinations, sortedness
onceover-mines     the model and statistical passes: encoding,
                   predictability screen (histogram GBM), decision
                   tables, PRISM, dependence measures, arithmetic
                   identities, local linear regimes, nominal
                   associations, date/sequence intelligence
onceover-report    finding types, evidence, consolidation and selection,
                   JSON / compact JSON / budgeted JSON / Markdown /
                   digest / brief rendering, lookup, filtering, and
                   report diffing (no polars dependency)
onceover           the modularized pipeline (pipeline/ stage modules),
                   budget tiers, detector registry, frame spec inference
                   and hierarchy discovery (spec.rs), multi-table
                   foreign-key analysis (many.rs), coverage
onceover-py        thin pyo3 bindings exposing profile, infer_spec,
                   profile_many, partition_scagnostics, and ProfileReport
```

The Python package adds only Polars-expression verification and check export
(`verify`, `rows`, `to_checks`), Altair HTML rendering (`to_html`), file/lazy
source loading, and CLI argument dispatch.

`onceover-report` depends on neither Polars nor the mining crates. It defines
the finding schema and serialization; the pipeline and detectors depend on
that schema. A finding's category describes the property found; its
`evidence.method.detector` identifies the detector that found it.

## The pipeline

The `profile` pipeline runs these passes in order:

1. **Extraction** — the polars `DataFrame` becomes a `TypedTable` of
   compact typed columns (zero-copy where the dtype allows it).
2. **Column cards** — exact per-column statistics, roles, concentration,
   format patterns; the card findings are emitted here.
3. **Row duplicates** — exact, with the logical shortcut when a proven
   key makes duplicates impossible.
4. **Keys** — single-column uniqueness from the cards' exact counts,
   composite-key screening and verification.
5. **Functional dependencies** — screen, lattice, verify; approximate
   dependencies and suppression.
6. **Sortedness and date gaps** — exact scans.
7. **Encoding** — the shared matrix for every model pass.
8. **Predictability screen** — every eligible column as a target.
9. **Decision tables, PRISM, regimes** — the model follow-ups, reading
   the screen's shortlists and fit in that order.
10. **Nominal associations** — pairwise value implications over the
    encoded matrix.
11. **Dependence and identities** — pairwise statistics, then affine
    probes with full-row verification.
12. **Rule evaluation** — the shared predicate evaluator over every
    conditional-rule candidate.
13. **Consolidation and selection** — equivalent statements merge,
    report-wide limits apply, ids are hashed from the canonical
    statement.

Between passes a deadline check (when one is set) records skipped work
in coverage. Inside the dependence screen, the hotspots scan, the
discontinuities pass, the surprises scan, and the duplicate-encodings
scan the same deadline is polled at work quanta, so a
long pass stops at a quantum boundary and its coverage entry states
how much of its search space was examined, rather than overrunning
the budget by its whole runtime.

Two of the wide-table searches bound their memory by construction.
The hotspots scan enumerates combinations with O(1) state — one
pass counts the examined cells for the Bonferroni denominator and
a second evaluates them — and the dependence screen addresses
pairs by ordinal, consuming them in chunks of 2^18 while survivor
selection keeps only a bounded heap plus the selected pairs. On a
5,000-column table, the two passes' peak incremental memory fell
from roughly 660–720 MiB to roughly 33–41 MiB under this scheme.
The discontinuities pass caches per-axis and per-pair scan data
(kept rows, standardized values, fit masks, backgrounds) under a
64 MiB byte budget, shared across targets; on the
taiwanese-bankruptcy corpus table that cache took the pass from
13,055 ms to 8,770 ms.

## The detector registry

Detectors are registered separately from the taxonomy: each declares an
id, the categories it can produce, and the detectors it requires as
resources. Selecting `["covering_rules", "association_rules"]` runs those detectors and their
required resources. Resource detectors may run without adding their own
findings. An empty selection still produces column cards and coverage.

In [1]:
import json

import onceover

for d in onceover.available_detectors():
    print(f"{d['id']:24s} -> {d['categories']}"
          + (f"   requires: {d['requires']}" if d["requires"] else ""))

column_profile           -> ['missingness', 'concentration', 'column_roles', 'formats']
row_duplicates           -> ['duplicates']
keys                     -> ['uniqueness']
pivot                    -> ['column_roles']
functional_dependencies  -> ['dependencies']
order_constraints        -> ['dependencies']
duplicate_encodings      -> ['column_roles']
sortedness               -> ['ordering']
granularity              -> ['formats']
sentinels                -> ['missingness']
boundary_mass            -> ['formats']
missingness              -> ['missingness']
content_anomalies        -> ['formats']
series                   -> ['gaps', 'changes']
row_consistency          -> ['duplicates', 'anomalies']   requires: ['numeric_dependence']
digit_fingerprints       -> ['formats']
date_gaps                -> ['gaps', 'anomalies']
expectations             -> ['gaps', 'ordering']
panel_coverage           -> ['gaps']
group_contrast           -> ['associations']
level_contrasts          -> ['associa

Rust extensions implement the `Detector` trait with a
`DetectorDescriptor` and a `discover(&DetectorContext)` returning
finding candidates. The pipeline validates their output categories,
subjects, and statement definitions; invalid results are rejected and reported in coverage.

## Budget tiers

`TierConfig` sets the search and sample limits for each budget. Coverage
reports the work performed under those limits:


In [2]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Setting': ['Exact passes (cards, duplicates, keys, sortedness, date gaps)', 'Composite keys', 'FD: max LHS size / verification rows', 'Relationship sample rows / per-model predictor limit', 'Predictability screen rows / rounds', 'Decision-table / covering-rule targets', 'Regimes: targets / rows / depth', 'Dependence pairs (capped)', 'Identity candidate pairs', 'Association findings'], '`quick`': ['full', '—', '—', '—', '—', '—', '—', '—', '—', '—'], '`standard`': ['full', 'pairs, verify ≤ 16', '2 / 200,000', '20,000 / 48', '5,000 / 30', '6 / 6', '4 / 5,000 / 3', '496', '100', '20'], '`deep`': ['full', 'pairs + triples, verify ≤ 24', '3 / 500,000', '50,000 / 64', '10,000 / 40', '10 / 10', '8 / 10,000 / 4', '2,016', '200', '40']})
(GT(_doc_table)
 .fmt_markdown(columns=['Setting', '`quick`', '`standard`', '`deep`'])
 .tab_header(title='Budget tiers')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


GT(_tbl_data=shape: (10, 4)
┌─────────────────────────────────┬─────────┬────────────────────┬──────────────────────────────┐
│ Setting                         ┆ `quick` ┆ `standard`         ┆ `deep`                       │
│ ---                             ┆ ---     ┆ ---                ┆ ---                          │
│ str                             ┆ str     ┆ str                ┆ str                          │
╞═════════════════════════════════╪═════════╪════════════════════╪══════════════════════════════╡
│ Exact passes (cards, duplicate… ┆ full    ┆ full               ┆ full                         │
│ Composite keys                  ┆ —       ┆ pairs, verify ≤ 16 ┆ pairs + triples, verify ≤ 24 │
│ FD: max LHS size / verificatio… ┆ —       ┆ 2 / 200,000        ┆ 3 / 500,000                  │
│ Relationship sample rows / per… ┆ —       ┆ 20,000 / 48        ┆ 50,000 / 64                  │
│ Predictability screen rows / r… ┆ —       ┆ 5,000 / 30         ┆ 10,000 / 40                  │
│ Decision-table / covering-rule… ┆ —       ┆ 6 / 6              ┆ 10 / 10                      │
│ Regimes: targets / rows / dept… ┆ —       ┆ 4 / 5,000 / 3      ┆ 8 / 10,000 / 4               │
│ Dependence pairs (capped)       ┆ —       ┆ 496                ┆ 2,016                        │
│ Identity candidate pairs        ┆ —       ┆ 100                ┆ 200                          │
│ Association findings            ┆ —       ┆ 20                 ┆ 40                           │
└─────────────────────────────────┴─────────┴────────────────────┴──────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x7a24f84270e0>, _boxhead=Boxhead([ColInfo(var='Setting', type=<ColInfoTypeEnum.default: 1>, column_label='Setting', column_align='left', column_width=None, stub_level=None), ColInfo(var='`quick`', type=<ColInfoTypeEnum.default: 1>, column_label='`quick`', column_align='left', column_width=None, stub_level=None), ColInfo(var='`standard`', type=<ColInfoTypeEnum.default: 1>, column_label='`standard`', column_align='left', column_width=None, stub_level=None), ColInfo(var='`deep`', type=<ColInfoTypeEnum.default: 1>, column_label='`deep`', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x7a24e0593c50>, _spanners=Spanners([]), _heading=Heading(title='Budget tiers', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7a24e2fb4b00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7a24e0b63f50>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Setting', 

`quick` runs cards, duplicate checks, single-column key checks, sortedness,
and date analysis; it skips relationship models. `sample_rows`,
`max_predictors`, and `max_pairs` change the sampled work, while the full-row
checks remain unchanged
([coverage](../usage/coverage.ipynb)).
